# SimCLR — learn features with no labels

**Foundation-model recipe** (supports track C and lesson on self-supervision).

Self-supervised contrastive learning is how backbones like SimCLR/DINO/CLIP are pretrained: pull two augmentations of the same image together, push different images apart (**NT-Xent**). We then freeze the encoder and train a **linear probe** to show the features became useful — beating a random-init encoder.

> CPU is fine (small synthetic images).

In [ ]:
import os, torch, torch.nn as nn, torch.nn.functional as F, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu"
STEPS = int(os.environ.get("STEPS", 800)); C = 6; S = 20

## 1 · Synthetic images — 6 shape classes at random position/scale (hard for raw features)

In [ ]:
def gen(n):
    x = torch.zeros(n, 1, S, S); y = torch.randint(0, C, (n,))
    base = torch.linspace(-1, 1, S)
    for i in range(n):
        cls = y[i].item(); cx, cy = (torch.rand(2) - 0.5) * 0.7; r = (0.25 + 0.3 * torch.rand(1)).item()
        gx, gy = torch.meshgrid(base - cx, base - cy, indexing="ij")
        if cls == 0: m = (gx ** 2 + gy ** 2) < r ** 2                                  # disc
        elif cls == 1: m = (gx.abs() < r) & (gy.abs() < r)                              # square
        elif cls == 2: m = (gx.abs() + gy.abs()) < r                                    # diamond
        elif cls == 3: m = ((gx ** 2 + gy ** 2) < r ** 2) & ((gx ** 2 + gy ** 2) > (0.55 * r) ** 2)  # ring
        elif cls == 4: m = (gx.abs() < 0.22 * r) | (gy.abs() < 0.22 * r)               # cross
        else: m = (gx.abs() < 0.18 * r)                                                 # bar
        x[i, 0] = m.float()
    x = (x + 0.1 * torch.randn_like(x)).clamp(0, 1)
    return x, y
def augment(x):
    x = x + 0.15 * torch.randn_like(x)
    if torch.rand(1) < 0.5: x = x.flip(-1)
    return (x * (0.6 + 0.8 * torch.rand(1))).clamp(0, 1)
xb, yb = gen(6)
plt.imshow(torch.cat([xb[i, 0] for i in range(6)], 1)); plt.title("synthetic shapes"); plt.axis("off"); plt.show()

## 2 · Encoder + projection head; NT-Xent contrastive loss

In [ ]:
def make_enc():
    return nn.Sequential(nn.Conv2d(1, 16, 3, 2, 1), nn.ReLU(), nn.Conv2d(16, 32, 3, 2, 1), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten()).to(device)   # -> 32-d feature
enc = make_enc(); proj = nn.Sequential(nn.Linear(32, 32), nn.ReLU(), nn.Linear(32, 16)).to(device)
def nt_xent(z, tau=0.5):
    z = F.normalize(z, dim=1); B = z.shape[0] // 2
    sim = z @ z.T / tau; sim.fill_diagonal_(-9e9)
    targets = torch.arange(2 * B, device=device); targets = (targets + B) % (2 * B)
    return F.cross_entropy(sim, targets)

## 3 · Pretrain — contrastive, no labels

In [ ]:
opt = torch.optim.Adam([*enc.parameters(), *proj.parameters()], 1e-3); hist = []
for step in range(STEPS + 1):
    x, _ = gen(64); v1 = augment(x).to(device); v2 = augment(x).to(device)
    z = proj(torch.cat([enc(v1), enc(v2)], 0)); loss = nt_xent(z)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % max(1, STEPS // 10) == 0: hist.append((step, round(loss.item(), 3))); print(f"step {step:4d}  NT-Xent {loss.item():.3f}")

## 4 · Linear probe — SimCLR features vs. a random encoder

In [ ]:
def probe(encoder, ntr=40):                       # few-shot probe: where self-supervision wins
    encoder.eval(); xtr, ytr = gen(ntr); xte, yte = gen(600)
    with torch.no_grad(): ftr = encoder(xtr.to(device)); fte = encoder(xte.to(device))
    clf = nn.Linear(32, C).to(device); o = torch.optim.Adam(clf.parameters(), 1e-2)
    for _ in range(300): o.zero_grad(); F.cross_entropy(clf(ftr), ytr.to(device)).backward(); o.step()
    return (clf(fte).argmax(1).cpu() == yte).float().mean().item()
simclr_acc = probe(enc); rand_acc = probe(make_enc())
print(f"linear-probe accuracy — SimCLR {simclr_acc:.3f}  vs  random encoder {rand_acc:.3f}")
fig, ax = plt.subplots(figsize=(5, 3.4)); ax.bar(["random", "SimCLR"], [rand_acc, simclr_acc], color=["C7", "C0"])
ax.set_ylim(0, 1); ax.set_ylabel("probe accuracy"); ax.set_title("self-supervision learns useful features"); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/C_simclr_pretrain/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/C_simclr_pretrain"; os.makedirs(run, exist_ok=True)
torch.save(enc.state_dict(), f"{run}/encoder.pt")
json.dump({"nt_xent": hist, "probe_simclr": simclr_acc, "probe_random": rand_acc}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

### Where to go next
- This is the recipe behind **SimCLR / MoCo / DINO**; CLIP adds text as the other view.
- Pretrain on real (unlabeled) egocentric frames, then probe/fine-tune for actions (track C).